# بول — Urdu TTS Backend on Google Colab
**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

## One-time setup (do this in Google Drive before running cells)
1. In Google Drive, right-click the shared **`urdu_tts_training`** folder → **"Add shortcut to My Drive"**
2. Make sure `MyDrive/FYP_Models/pretrained_models/` still exists (ve, s3gen, t3_cfg, conds.pt)
3. Make sure your ngrok token is ready: https://dashboard.ngrok.com/get-started/your-authtoken

## Model sources
- **Standard pretrained models** (ve, s3gen, t3_cfg, conds): `MyDrive/FYP_Models/pretrained_models/`
- **Extended tokenizer** (2454-vocab Urdu): `urdu_tts_training/repos/chatterbox-finetuning/pretrained_models/tokenizer.json`
- **New fine-tuned weights**: `urdu_tts_training/repos/chatterbox-finetuning/chatterbox_output/t3_finetuned.safetensors`

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────
!pip install -q fastapi "uvicorn[standard]" sqlalchemy "passlib[bcrypt]" "python-jose[cryptography]" python-multipart soundfile pyngrok nest_asyncio chatterbox-tts==0.1.2
!pip uninstall -y torchvision
print('Dependencies installed.')

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── Cell 4: Clone GitHub repo ─────────────────────────────────────────────
import os

if not os.path.exists('/content/FYP'):
    !git clone https://github.com/Bilal-Shinwari/FYP-Project.git /content/FYP
else:
    !cd /content/FYP && git pull

print('Repo ready.')

In [ ]:
# ── Cell 5: Copy model weights from Google Drive ──────────────────────────
import shutil, os

DRIVE = '/content/drive/MyDrive/urdu_tts_training/repos/chatterbox-finetuning'
FINETUNE_DIR = '/content/FYP/t3_finetuned_model/chatterbox-finetuning'

dst_pretrained = f'{FINETUNE_DIR}/pretrained_models'
dst_weights    = f'{FINETUNE_DIR}/t3_finetuned.safetensors'
dst_ref        = f'{FINETUNE_DIR}/speaker_reference/2.wav'

# 1. Pretrained models (ve, s3gen, t3_cfg, conds.pt, tokenizer.json)
if not os.path.exists(dst_pretrained):
    print('Copying pretrained_models...')
    shutil.copytree(f'{DRIVE}/pretrained_models', dst_pretrained)
else:
    print('pretrained_models already present.')

# 2. Fine-tuned T3 weights (2.14 GB — takes ~1 min)
print('Copying t3_finetuned.safetensors...')
shutil.copy2(f'{DRIVE}/chatterbox_output/t3_finetuned.safetensors', dst_weights)
print('Done.')

# 3. Speaker reference wav
os.makedirs(f'{FINETUNE_DIR}/speaker_reference', exist_ok=True)
if not os.path.exists(dst_ref):
    src_ref = f'{DRIVE}/speaker_reference/2.wav'
    if os.path.exists(src_ref):
        shutil.copy2(src_ref, dst_ref)
        print('Speaker reference copied.')
    else:
        print('WARNING: speaker_reference/2.wav not found — check Drive path manually.')

print('\nFile check:')
print('  t3_finetuned.safetensors        :', os.path.exists(dst_weights))
print('  pretrained_models/tokenizer.json:', os.path.exists(f'{dst_pretrained}/tokenizer.json'))
print('  speaker_reference/2.wav         :', os.path.exists(dst_ref))

In [ ]:
# ── Cell 6: Configure ngrok ───────────────────────────────────────────────
# Get a FREE token at: https://dashboard.ngrok.com/get-started/your-authtoken
# Sign up is free, takes 30 seconds

NGROK_TOKEN = ''  # ← paste your token here

from pyngrok import ngrok
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    print('ngrok token set.')
else:
    print('WARNING: No ngrok token. Get one free at https://dashboard.ngrok.com')

In [ ]:
# ── Cell 7: Start the FastAPI backend ─────────────────────────────────────
import os, sys, subprocess, time, nest_asyncio
from pyngrok import ngrok

nest_asyncio.apply()

FINETUNE_DIR = '/content/FYP/t3_finetuned_model/chatterbox-finetuning'
BACKEND_DIR  = '/content/FYP/text-to-speech/tts-backend'

os.makedirs(f'{BACKEND_DIR}/static', exist_ok=True)

env = {
    **os.environ,
    'FINETUNE_DIR': FINETUNE_DIR,
    'ALLOW_ALL_ORIGINS': '1',        # allow frontend on any domain
}

print('Starting FastAPI server...')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=BACKEND_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Stream startup logs until model is loaded (or 5 min timeout)
import threading

model_ready = threading.Event()

def stream_logs():
    for line in server.stdout:
        print(line, end='')
        if 'Model loaded successfully' in line or 'Application startup complete' in line:
            model_ready.set()

t = threading.Thread(target=stream_logs, daemon=True)
t.start()

print('Waiting for model to load (30-90s on GPU)...')
model_ready.wait(timeout=300)

# Open ngrok tunnel
tunnel = ngrok.connect(8000)
BACKEND_URL = tunnel.public_url

print('\n' + '='*60)
print('✅ BACKEND IS LIVE')
print(f'   URL: {BACKEND_URL}')
print('='*60)
print('\n📱 To connect your local frontend:')
print(f'   1. Open your browser at http://localhost:5173')
print(f'   2. Open browser DevTools console (F12)')
print(f'   3. Paste this command:')
print(f"\n   localStorage.setItem('api_base', '{BACKEND_URL}')")
print(f'\n   4. Refresh the page — done!')

In [ ]:
# ── Cell 8: Quick API test ─────────────────────────────────────────────────
import requests

# Register test user
r = requests.post(f'{BACKEND_URL}/register', json={
    'username': 'colabtest', 'email': 'colab@fyp.com', 'password': 'test1234'
})
print('Register:', r.status_code, r.json())

# Login
r = requests.post(f'{BACKEND_URL}/token', data={
    'username': 'colabtest', 'password': 'test1234'
})
token = r.json()['access_token']
print('Login: OK, token received')

# TTS test
print('\nSending TTS request...')
r = requests.post(
    f'{BACKEND_URL}/tts_simple',
    headers={'Authorization': f'Bearer {token}'},
    data={'text': 'یہ ایک ٹیسٹ ہے۔'}
)
print(f'TTS response: HTTP {r.status_code}, {len(r.content)} bytes')

if r.status_code == 200:
    with open('/content/test_output.wav', 'wb') as f:
        f.write(r.content)
    import soundfile as sf
    data, sr = sf.read('/content/test_output.wav')
    print(f'✅ Audio OK — {round(len(data)/sr, 2)}s at {sr}Hz')
    
    # Play in Colab
    from IPython.display import Audio, display
    display(Audio('/content/test_output.wav'))
else:
    print('Error:', r.json())

## Keep this tab open
The backend runs as long as this Colab session is active.  
Free Colab sessions last ~12 hours.  

To reconnect your frontend to a new session URL, run Cell 7 again and repeat the `localStorage.setItem` step.